In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler
import yfinance as yf
import tiktoken
from datasets import load_dataset
import json
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
cpu available


## Class Definitions:

    - AttentionNode: self attention head written as proof of concept
    - MaskedAttentionNode: maksed self attention head written as proof of concept
    - MultiHeadAttention: Multi head attention, gathering individual AttentionNode
    - MaskedMultiHeadAttentionNode: Mixes multi head attention into a single, more efficient, structure
    - TransformerBlock: Combines MaskedMultiHeadAttentionNode and a feedforward perceptron
    - Decoder: Layers Embeddings and TransformerBlocks to learn sequence generation

The model itself only uses the MaskedMultiHeadAttentionNode, TransformerBlock, and Decoder classes. The others are included for pedagogical value. 

In [14]:
class AttentionNode(nn.Module):
    def __init__(self, d_in, d_qk, d_v):
        super().__init__()

        self.Wq = nn.Linear(d_in, d_qk)
        self.Wk = nn.Linear(d_in, d_qk)
        self.Wv = nn.Linear(d_in, d_v)

        self.scale = d_qk ** 0.5

    def forward(self, x):
        Q = self.Wq(x) # batch N d_qk
        K = self.Wk(x) # batch N d_qk
        V = self.Wv(x) # batch N d_v
        score = torch.matmul(Q, K.transpose(-2, -1)) # batch N N
        score = score / self.scale

        weights = F.softmax(score, dim=-1)
        out = torch.matmul(weights, V) # batch N d_v

        return out, weights

class MaskedAttentionNode(nn.Module):
    def __init__(self, d_in, d_qk, d_v, dropout = 0.2):
        super().__init__()

        self.Wq = nn.Linear(d_in, d_qk)
        self.Wk = nn.Linear(d_in, d_qk)
        self.Wv = nn.Linear(d_in, d_v)
        self.dropout = nn.Dropout(dropout)

        self.scale = d_qk ** 0.5

    def forward(self, x):
        Q = self.Wq(x) # batch N d_qk
        K = self.Wk(x) # batch N d_qk
        V = self.Wv(x) # batch N d_v
        score = torch.matmul(Q, K.transpose(-2, -1)) # batch N N
        score = score / self.scale

        N = x.size(1)
        mask = torch.triu(torch.ones(N, N, device=x.device), diagonal=1).bool()
        score = score.masked_fill(mask, float('-inf'))

        weights = self.dropout(F.softmax(score, dim=-1))
        out = torch.matmul(weights, V) # batch N d_v

        return out, weights
    
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        d_head = int(d_model / num_heads)
        super().__init__()
        self.heads = nn.ModuleList([
            MaskedAttentionNode(d_model, d_head, d_head) for _ in range(num_heads)
        ])

        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        head_out, head_w = zip(*[head(x) for head in self.heads])

        out = torch.cat(head_out, dim=-1)  # (B, N, num_heads * d_v)
        out = self.Wo(out)                     # (B, N, d_in)
        
        weights = torch.stack(head_w, dim=1)  # (B, num_heads, N, N)
        return out, weights

class MaskedMultiHeadAttentionNode(nn.Module):
    def __init__(self, d_model, num_heads, dropout = 0.2):
        assert d_model % num_heads == 0
        super().__init__()

        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.dropout = nn.Dropout(dropout)

        self.num_heads = num_heads
        self.d_head = int(d_model / num_heads)
        self.scale = self.d_head ** 0.5

    def forward(self, x):
        B, N, D = x.shape
        QKV = self.Wqkv(x)
        QKV = QKV.view(B, N, 3, self.num_heads, self.d_head)
        QKV = QKV.permute(2, 0, 3, 1, 4)  # (3, B, H, N, d_head)
        Q, K, V = QKV[0], QKV[1], QKV[2]

        score = (Q @ K.transpose(-2, -1)) # batch H N N
        score = score / self.scale

        N = x.size(1)
        mask = torch.triu(torch.ones(N, N, device=x.device), diagonal=1).bool()
        score = score.masked_fill(mask, float('-inf'))

        weights = self.dropout(F.softmax(score, dim=-1))
        out = torch.matmul(weights, V) # batch H N d_head

        out = out.transpose(1, 2).contiguous() # B N H d_head
        out = out.view(B, N, D)

        return out, weights

class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff, heads, layers, dropout = 0.2):
        super().__init__()
        self.attn = MaskedMultiHeadAttentionNode(d_model, heads)
        layers_list = [
            module
            for _ in range(layers)
            for module in (
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_ff, d_model),
            )
        ]
        self.ff = nn.Sequential(*layers_list)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x))[0])
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, transformer_layers, ctx_window):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.position = nn.Embedding(ctx_window, d_model)
        #self.mode_embed = nn.Embedding(num_modes, d_model)
        self.transformers = nn.Sequential(
            *[TransformerBlock(d_model, d_ff, num_heads, transformer_layers) for i in range(num_layers)]
        )
        self.ln = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, vocab_size, bias = False)

    def forward(self, x):#, modes):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device)

        embedded = self.embed(x) + self.position(positions).unsqueeze(0)# + self.mode_embed(modes).unsqueeze(1) # B N d_model
        transformed = self.transformers(embedded)
        norm = self.ln(transformed)
        pred = self.unembed(norm)

        return pred

## Datasets:

First, we use the tinyshakespeare dataset at a character level for model validation. When learned, the model can generate Shakespeare-like writing, sometimes making up words, but coherent enough to replicate the structure and voice.

Next, we use the `gpt-2` tokenizer to read through wikipedia data. The adidtional tokens are added to help the decoder parse the dataset's special tokens. The context window is embedded into the format of the dataset.

In [11]:
seq_size = 128

text = open("tinyshakespeare.txt", "r").read()

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {c : i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)} 

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

def compile(x, seq_len):
        x_sequences = []
        y_sequences = []

        for i in tqdm(range(0, len(x) - seq_len - 1, int(seq_len/2))):
            x_sequences.append(x[i:i+seq_len])
            y_sequences.append(x[i+1:i+seq_len+1])

        x_sequences = np.array(x_sequences)
        y_sequences = np.array(y_sequences)

        return torch.tensor(x_sequences), torch.tensor(y_sequences)

x_data, y_data = compile(encode(text), seq_size)
print(x_data.shape)
assert x_data.shape == y_data.shape, f"make sure x shape {x_data.shape} matches y shape {y_data.shape}"

print(x_data.shape[0])

split = int(0.8 * x_data.shape[0])
x_train, y_train = x_data[:split], y_data[:split]
x_val, y_val = x_data[split:], y_data[split:]

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_dataset = TensorDataset(x_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

100%|██████████| 17427/17427 [00:00<00:00, 51462.46it/s]

torch.Size([17427, 128])
17427


In [ ]:
def compile(x, seq_len):
        x_sequences = []
        y_sequences = []

        for i in tqdm(range(0, len(x) - seq_len - 1, int(seq_len/2))):
            x_sequences.append(x[i:i+seq_len])
            y_sequences.append(x[i+1:i+seq_len+1])

        x_sequences = np.array(x_sequences)
        y_sequences = np.array(y_sequences)

        return torch.tensor(x_sequences), torch.tensor(y_sequences)

seq_size = 128
dl = False

base_enc = tiktoken.get_encoding("gpt2")

custom_tokens = {
    "<USER>",
    "<ASSISTANT>",
    "_START_ARTICLE_",
    "_START_SECTION_",
    "_START_PARAGRAPH_"
}

custom_token_ids = {
    token: base_enc.n_vocab + i for i, token in enumerate(custom_tokens)
}

enc = tiktoken.Encoding(
    name="gpt2_custom",
    pat_str=base_enc._pat_str,
    mergeable_ranks=base_enc._mergeable_ranks,
    special_tokens={**base_enc._special_tokens, **custom_token_ids},
)

special_tokens_set = set(custom_tokens) | {"<|endoftext|>"}

encode = enc.encode
decode = enc.decode
vocab_size = enc.n_vocab
print(vocab_size)

if dl:
    wiki = load_dataset("wiki40b", "en", split="train")
    text = "\n".join([x['text'] for x in wiki.take(10000)])  # grab first 10k articles
    with open("datasets/wiki.txt", "w") as f:
        f.write(text)


text = open("datasets/wiki.txt", "r").read()
text = re.sub(r'<[^>]+>', '', text)        # strip any html tags
text = re.sub(r'\s+', ' ', text)           # normalize whitespace
text = text.strip()

tokens = encode(text, allowed_special=special_tokens_set)
x_data, y_data = compile(tokens, seq_size)
print(decode(x_data[0].tolist()))
print(decode(y_data[0].tolist()))
assert x_data.shape == y_data.shape, f"make sure x shape {x_data.shape} matches y shape {y_data.shape}"

split = int(0.8 * x_data.shape[0])
x_train, y_train = x_data[:split], y_data[:split]
x_val, y_val = x_data[split:], y_data[split:]

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_dataset = TensorDataset(x_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

50262


100%|██████████| 75155/75155 [00:00<00:00, 107564.42it/s]


tensor([50261,  6184,   223, 12397, 27974,   282, 10205,  4106,   220, 50258])
_START_ARTICLE_ Ági Szalóki _START_SECTION_ Life _START_PARAGRAPH_ She started singing as a toddler, considering Márta Sebestyén a role model. Her musical background is traditional folk music; she first won recognition for singing with Ökrös in a traditional folk style, and Besh o droM, a Balkan gypsy brass band. With these ensembles she toured around the world from the Montreal Jazz Festival, through Glastonbury Festival to the Théatre de la Ville in Paris, from New York to Beijing._NEWLINE_Since 2005, she began to pursue her solo career and explore various genres
 Ági Szalóki _START_SECTION_ Life _START_PARAGRAPH_ She started singing as a toddler, considering Márta Sebestyén a role model. Her musical background is traditional folk music; she first won recognition for singing with Ökrös in a traditional folk style, and Besh o droM, a Balkan gypsy brass band. With these ensembles she toured around the world 

## Model Training

In [ ]:
whereswaldo = Decoder(
    vocab_size=vocab_size,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=8,
    transformer_layers=2,
    ctx_window=seq_size,
).to(device)

optimizer = torch.optim.AdamW(whereswaldo.parameters(), lr=5e-3, weight_decay=0.1)

loss_fn = F.cross_entropy
# def loss_fn(pred, y, mask):
#     loss_fn = F.cross_entropy(reduction="none")
#     loss_unwrap = loss_fn(pred.view(-1, vocab_size), # BxN vocab_size
#                            y.view(-1)) # BxN
#     mask_unwrap = mask.view(-1) # BxN
#     return (loss_unwrap * mask_unwrap).sum() / loss_unwrap.sum()

epochs = 15

train_losses = []
val_losses = []
for i in range(epochs):
    whereswaldo.train()
    train_loss = 0
    for train_batch in tqdm(train_loader):
        x, y = train_batch
        x = x.to(device)
        y = y.to(device)
        pred = whereswaldo(x)
        loss = loss_fn(pred.view(-1, vocab_size), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(whereswaldo.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss
    train_loss /= len(train_loader)
    
    whereswaldo.eval()
    with torch.no_grad():
        val_loss = 0
        for val_batch in val_loader:
            x, y = val_batch
            x = x.to(device)
            y = y.to(device)
            pred = whereswaldo(x)
            val_loss += loss_fn(pred.view(-1, vocab_size), y.view(-1))
    
    val_loss /= len(val_loader)

    print(f"Epoch {i+1}: train loss = {train_loss:.6f}, val loss = {val_loss:.6f}")
    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot()

ax.plot(train_losses, label="Train")
ax.plot(val_losses, label="Validation")
ax.legend()
ax.set_title("Training vs Validation Loss")
plt.show()


  0%|          | 0/940 [00:00<?, ?it/s]/home/park/proj/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  0%|          | 0/940 [00:09<?, ?it/s]


KeyboardInterrupt: 

In [6]:
save = False
load = False

if save:
    torch.save({
        'model_state': whereswaldo.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'epoch': i,
        'val_loss': val_loss,
    }, 'whereswaldo.pt')

    tokenizer = {'stoi': stoi, 'itos': {str(k): v for k, v in itos.items()}}
    with open('tokenizer.json', 'w') as f:
        json.dump(tokenizer, f)


if load:
    checkpoint = torch.load('whereswaldo.pt')
    whereswaldo.load_state_dict(checkpoint['model_state'])

    # load
    with open('tokenizer.json', 'r') as f:
        tokenizer = json.load(f)
    stoi = tokenizer['stoi']
    itos = {int(k): v for k, v in tokenizer['itos'].items()}

In [51]:
def query(model : Decoder, prompt, generation_len = 2 * seq_size, live = True):
    model.eval()
    tokens = torch.tensor(encode(prompt)).unsqueeze(0).to(device)
    if live: print(prompt, end="")


    for i in range(generation_len):
        tokens_cropped = tokens[:, -seq_size:]
        scores = model(tokens_cropped)[:, -1, :]
        probs = F.softmax(scores, dim = -1)
        char = torch.multinomial(probs, num_samples=1)
        if live: print(decode([char[0].item()]), end = "") 
        tokens = torch.cat([tokens, char], dim = 1)

    return decode(tokens[0].tolist())

res = query(whereswaldo, "ROMEO: I'm genuinely pissing myself my love. ", 1024)

ROMEO: I'm genuinely pissing myself my love. ulsive Gamblers, Prade began to be a bad commercial success and punk rock efforts, used in powerful rock metal sound with instruments finding instrument. Whether this sound was merely listening to 188 pounds – a passion for his brilliant performance equipment was first danced to create and jazz harmony album as well as the background to Anthony Lydney and Benjamin Miller, Peace & Wilraham Branttert was re-released. On drug in July 1971 for the first time, Causes and asked the online issues of alternative experience._NEWLINE_In October 1959 Recreations of the summer of 1980, EPs of acid (avanitation pictures are non-continent sections, significance, political and (agreed). From the wider homestate business and other stations in Monaco, so that it takes must be used roughly,000 years ago), allographic roads, although does not depend upon adopting a ripen policy issues and decom/or newer deficiencies and domestic, which are also problematic for